In [1]:
pip install gradio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image
import gradio as gr
import numpy as np
import cv2
from skimage.measure import shannon_entropy
import pennylane as qml
import timm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ================= MODEL ================= #
class QuantumLayer(nn.Module):
    def __init__(self, n_qubits=6, n_layers=3):
        super().__init__()
        dev = qml.device('default.qubit', wires=n_qubits)

        @qml.qnode(dev, interface='torch', diff_method='backprop')
        def circuit(inputs, weights):
            for i in range(n_qubits):
                qml.RY(inputs[i], wires=i)
            for l in range(n_layers):
                for i in range(n_qubits):
                    qml.RX(weights[l, i, 0], wires=i)
                    qml.RY(weights[l, i, 1], wires=i)
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i+1])
                qml.CNOT(wires=[n_qubits-1, 0])
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.circuit = circuit
        self.q_params = nn.Parameter(0.1 * torch.randn(n_layers, n_qubits, 2))

    def forward(self, x):
        return torch.stack([
            torch.stack(self.circuit(x[b], self.q_params)).float()
            for b in range(x.shape[0])
        ])

class QuantumMixingBlock(nn.Module):
    def __init__(self, in_dim, n_qubits=6):
        super().__init__()
        self.pre = nn.Linear(in_dim, n_qubits)
        self.bn = nn.BatchNorm1d(n_qubits)
        self.quantum = QuantumLayer(n_qubits=n_qubits)
        self.post = nn.Linear(n_qubits, in_dim)

    def forward(self, x):
        q_in = self.bn(self.pre(x))
        q_in = F.normalize(q_in, dim=1) * torch.pi
        q_out = self.quantum(q_in)
        return self.post(q_out) + x

class HybridViT(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = timm.create_model('vit_tiny_patch16_224', pretrained=False, num_classes=0)
        embed_dim = self.backbone.num_features
        self.qblock = QuantumMixingBlock(embed_dim)
        self.head = nn.Sequential(nn.LayerNorm(embed_dim), nn.Linear(embed_dim, num_classes))

    def forward(self, x):
        return self.head(self.qblock(self.backbone(x)))

# ================= LOAD MODEL ================= #
ckpt = torch.load('qvit_final_model.pth', map_location=DEVICE)
model = HybridViT()
model.load_state_dict(ckpt['model_state'])
model.to(DEVICE)
model.eval()

class_to_idx = ckpt['class_to_idx']
threshold = ckpt['threshold']

# ================= TRANSFORM ================= #
infer_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# ================= METRICS ================= #
def compute_metrics(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    brightness = np.mean(gray)
    contrast = np.std(gray)
    entropy = shannon_entropy(gray)
    return brightness, contrast, entropy

# ================= CLAHE ================= #
def apply_clahe(image):
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(2.0, (8,8))
    cl = clahe.apply(l)
    return cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2RGB)

# ================= BLOB DETECTION + HEAT ================= #
def detect_blobs(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Detect dark blobs
    _, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY_INV)

    kernel = np.ones((5,5), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    output = image.copy()

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if 50 < area < 5000:
            x, y, w, h = cv2.boundingRect(cnt)

            roi = gray[y:y+h, x:x+w]

            # Heatmap (RED-YELLOW style)
            heat = cv2.normalize(roi, None, 0, 255, cv2.NORM_MINMAX)
            heatmap = cv2.applyColorMap(heat, cv2.COLORMAP_HOT)

            roi_color = output[y:y+h, x:x+w]
            blended = cv2.addWeighted(roi_color, 0.5, heatmap, 0.6, 0)

            output[y:y+h, x:x+w] = blended

            # Draw box
            cv2.rectangle(output, (x, y), (x+w, y+h), (0,255,0), 2)

    return output

# ================= MAIN ================= #
def process(image):
    img = np.array(image)

    brightness, contrast, entropy = compute_metrics(img)
    enhanced = apply_clahe(img)

    # Prediction
    pil_img = Image.fromarray(enhanced)
    x = infer_tf(pil_img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]

    abnormal_idx = class_to_idx['abnormal']
    prob_abnormal = probs[abnormal_idx].item()
    prob_normal = probs[1 - abnormal_idx].item()

    if prob_abnormal >= threshold:
        label = "🔴 ABNORMAL"
        confidence = prob_abnormal
    else:
        label = "🟢 NORMAL"
        confidence = prob_normal

    # Blob + heat
    blob_img = detect_blobs(enhanced)

    return (
        img,
        enhanced,
        blob_img,
        f"{label} ({confidence*100:.2f}%)",
        f"{brightness:.2f}",
        f"{contrast:.2f}",
        f"{entropy:.2f}"
    )

# ================= UI ================= #
with gr.Blocks(css="""
body {background-color: #0f172a; color: white;}
h1 {text-align: center; font-size: 42px; color: #38bdf8;}
h2 {text-align: center; color: #cbd5f5;}
.gr-button {background: linear-gradient(90deg,#2563eb,#38bdf8); color:white;}
""") as demo:

    gr.Markdown("# 💓 Fetal Cardiac Analysis System")
    gr.Markdown("### QViT + CLAHE + Region Highlighting")

    input_img = gr.Image(type="numpy", label="Upload Ultrasound Image")

    with gr.Row():
        original = gr.Image(label="Original")
        enhanced = gr.Image(label="Enhanced")

    blob_img = gr.Image(label="Detected Cardiac Regions")

    prediction = gr.Textbox(label="Prediction")

    with gr.Row():
        b = gr.Textbox(label="Brightness")
        c = gr.Textbox(label="Contrast")
        e = gr.Textbox(label="Entropy")

    btn = gr.Button("🚀 Run Analysis")

    btn.click(
        process,
        inputs=input_img,
        outputs=[original, enhanced, blob_img, prediction, b, c, e]
    )

demo.launch()

C:\Users\nishu\AppData\Local\Temp\ipykernel_10228\2286609410.py:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load('qvit_final_model.pth', map_location=DEVI

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
